## 구글드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 데이터 분할 (train/ valid/ test split)

In [ ]:
import os
import shutil
import random
from collections import defaultdict

# 원본 데이터 경로
src_root = "/content/drive/MyDrive/salpim_project/00_data/RGB_selected"

# 저장 경로
dst_root = "/content/drive/MyDrive/salpim_project/01_dataset"

# split 비율
train_ratio = 0.7
valid_ratio = 0.15
test_ratio = 0.15

# 랜덤 고정
random.seed(42)

# 행동별 파일 저장
action_files = defaultdict(list)

video_exts = (".mp4", ".avi", ".mov", ".mkv")

# 전체 영상 탐색
for root, dirs, files in os.walk(src_root):

    for file_name in files:

        if not file_name.lower().endswith(video_exts):
            continue

        if not file_name.startswith("A"):
            continue

        try:
            action_num = int(file_name.split("_")[0][1:])
        except:
            continue

        full_path = os.path.join(root, file_name)

        action_files[action_num].append(full_path)

# 행동별 split
for action, file_list in action_files.items():

    random.shuffle(file_list)

    total = len(file_list)

    train_end = int(total * train_ratio)
    valid_end = train_end + int(total * valid_ratio)

    train_files = file_list[:train_end]
    valid_files = file_list[train_end:valid_end]
    test_files = file_list[valid_end:]

    split_dict = {
        "train": train_files,
        "valid": valid_files,
        "test": test_files
    }

    for split_name, split_files in split_dict.items():

        save_dir = os.path.join(
            dst_root,
            split_name,
            f"A{action:03d}"
        )

        os.makedirs(save_dir, exist_ok=True)

        for src_file in split_files:

            file_name = os.path.basename(src_file)

            dst_file = os.path.join(save_dir, file_name)

            shutil.copy2(src_file, dst_file)

print("데이터 분할 완료")

데이터 분할 완료


## 기본 전처리 (Frame Extraciton + Resize)

In [ ]:
import cv2
import os

# 입력 경로
input_root = "/content/drive/MyDrive/salpim_project/01_dataset"

# 출력 경로
output_root = "/content/drive/MyDrive/salpim_project/02_processed_frames"

# 추출 fps
target_fps = 10

# 입력 크기
img_size = 224

video_exts = (".mp4", ".avi", ".mov", ".mkv")

for split in ["train", "valid", "test"]:

    split_path = os.path.join(input_root, split)

    for action in os.listdir(split_path):

        action_path = os.path.join(split_path, action)

        for video_name in os.listdir(action_path):

            if not video_name.lower().endswith(video_exts):
                continue

            video_path = os.path.join(action_path, video_name)

            cap = cv2.VideoCapture(video_path)

            original_fps = cap.get(cv2.CAP_PROP_FPS)

            frame_interval = max(1, int(original_fps / target_fps))

            save_dir = os.path.join(
                output_root,
                split,
                action,
                video_name.split(".")[0]
            )

            os.makedirs(save_dir, exist_ok=True)

            frame_count = 0
            saved_count = 0

            while True:

                ret, frame = cap.read()

                if not ret:
                    break

                if frame_count % frame_interval == 0:

                    # resize
                    frame = cv2.resize(
                        frame,
                        (img_size, img_size)
                    )

                    save_path = os.path.join(
                        save_dir,
                        f"frame_{saved_count:04d}.jpg"
                    )

                    cv2.imwrite(save_path, frame)

                    saved_count += 1

                frame_count += 1

            cap.release()

            print(f"{video_name} 완료")

print("프레임 추출 완료")

A018_P211_G001_H120.mp4 완료
A018_P203_G001_H070.mp4 완료
A018_P201_G005_H070.mp4 완료
A018_P006_C0li_C001.mp4 완료
A018_P220_G001_H120.mp4 완료
A018_P205_G008_H070.mp4 완료
A018_P202_G002_H070.mp4 완료
A018_P226_G002_H070.mp4 완료
A018_P225_G001_H070.mp4 완료
A018_P206_G003_H120.mp4 완료
A018_P220_G002_H070.mp4 완료
A018_P018_C0li_C0li.mp4 완료
A018_P202_G003_H120.mp4 완료
A018_P007_C0li_C001.mp4 완료
A018_P204_G002_H070.mp4 완료
A018_P223_G002_H120.mp4 완료
A018_P001_C0li_C0li.mp4 완료
A018_P204_G012_H070.mp4 완료
A018_P215_G002_H120.mp4 완료
A018_P205_G004_H120.mp4 완료
A018_P204_G009_H120.mp4 완료
A018_P219_G002_H120.mp4 완료
A018_P204_G011_H070.mp4 완료
A018_P003_C0li_C0li.mp4 완료
A018_P002_C0li_C0li.mp4 완료
A018_P216_G001_H120.mp4 완료
A018_P202_G012_H120.mp4 완료
A018_P224_G001_H120.mp4 완료
A018_P213_G002_H070.mp4 완료
A018_P203_G006_H070.mp4 완료
A018_P230_G002_H070.mp4 완료
A018_P202_G008_H120.mp4 완료
A018_P204_G001_H070.mp4 완료
A018_P203_G008_H120.mp4 완료
A018_P208_G002_H070.mp4 완료
A018_P202_G009_H070.mp4 완료
A018_P008_C0li_C001.mp4 완료
A

## 데이터 증강

In [ ]:
from torchvision import transforms

# train augmentation
train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(p=0.5),

    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# valid / test
valid_test_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])